# Module 6: Measuring the Trend

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 7](../Beginner/Topic_07_Trend.md) said to look at several years
before judging a direction. This module puts a number on that direction, and,
more importantly, puts an interval around the number.

The interval is the part people leave out. "Down 5 percent a year" sounds
precise. It usually is not.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

## 2. Fit the line on the log scale

A straight line fitted to the **log** of a series is a constant **percentage**
change per period. A straight line fitted to the raw series is a constant number
of incidents per period, which is rarely what anyone means by a trend.

The slope comes out per month, so multiply by 12 and convert back from logs:

    percent per year = 100 × (exp(12 × slope) − 1)

In [ ]:
import statsmodels.formula.api as smf

g = final[final["agency_id"] == "A012"].copy()
g["rate"] = 100 * g["n_uof"] / g["n_arrests"]
g["t"] = np.arange(len(g))
g["mon"] = g["year_month"].str[5:7].astype(int)

whole = g[g["year_month"] <= "2025-12"]        # whole calendar years only

fit = smf.ols("np.log(rate) ~ t", data=whole).fit()
per_year = lambda b: 100 * (np.exp(12 * b) - 1)
lo, hi = fit.conf_int().loc["t"]

print(f"estimated change: {per_year(fit.params['t']):+.2f} percent a year")
print(f"95 percent interval: {per_year(lo):+.2f} to {per_year(hi):+.2f}")

The dataset was built with a decline of about **4.9 percent a year**, so the
estimate is close. Note the interval though: anywhere from about 3 to about 7
percent is consistent with seven years of monthly data from a large agency.

Anyone quoting the point estimate alone is quoting one number out of a range
two and a half points wide.

## 3. Month effects narrow the interval

Adding a term for each calendar month removes the seasonal swing from the
residuals. The slope barely moves, but the interval tightens, because the model
is no longer treating predictable seasonal variation as error.

In [ ]:
fit2 = smf.ols("np.log(rate) ~ t + C(mon)", data=whole).fit()
lo2, hi2 = fit2.conf_int().loc["t"]

print(f"without month effects: {per_year(fit.params['t']):+.2f} "
      f"[{per_year(lo):+.2f}, {per_year(hi):+.2f}]  width {per_year(hi)-per_year(lo):.2f}")
print(f"with month effects   : {per_year(fit2.params['t']):+.2f} "
      f"[{per_year(lo2):+.2f}, {per_year(hi2):+.2f}]  width {per_year(hi2)-per_year(lo2):.2f}")

## 4. The window does most of the work

In [ ]:
def slope(d, formula="np.log(rate) ~ t"):
    o = smf.ols(formula, data=d).fit()
    lo, hi = o.conf_int().loc["t"]
    return per_year(o.params["t"]), per_year(lo), per_year(hi)

windows = {
    "2024 and 2025 only": g[(g["year_month"] >= "2024-01") & (g["year_month"] <= "2025-12")],
    "2021 through 2023": g[(g["year_month"] >= "2021-01") & (g["year_month"] <= "2023-12")],
    "2019 through 2025": whole,
}
rows = {k: slope(v) for k, v in windows.items()}
rows["2019 through 2025, month effects"] = slope(whole, "np.log(rate) ~ t + C(mon)")

out = pd.DataFrame(rows, index=["estimate", "low", "high"]).T.round(2)
out["width"] = (out["high"] - out["low"]).round(2)
out

Every interval covers the truth of about 4.9 percent. What changes is how much
they say. Two years of data gives an interval roughly 30 points wide, which
includes a large improvement and a large deterioration at the same time. It is
compatible with almost anything.

**A short window does not give you a less accurate answer. It gives you an
answer that means nothing.**

## 5. Simpler alternatives, and when they are fine

The compound growth rate between the first and last year is easy to compute and
easy to explain.

In [ ]:
yearly = (whole.assign(year=whole["year_month"].str[:4])
          .groupby("year")
          .apply(lambda x: 100 * x["n_uof"].sum() / x["n_arrests"].sum()))

n = len(yearly) - 1
cagr = 100 * ((yearly.iloc[-1] / yearly.iloc[0]) ** (1 / n) - 1)

print(yearly.round(2).to_string())
print(f"\nchange per year from first to last: {cagr:+.2f} percent")
print(f"regression on all 84 months        : {per_year(fit.params['t']):+.2f} percent")

The two agree here. They will not always, because the first and last figures use
two numbers while the regression uses all of them. Use the simple version to
communicate and the regression to decide, and never report the simple version
without saying which two years it compares.

## 6. One slope assumes one story

A single straight line says the agency changed at a steady rate throughout. If
something happened part way through, that assumption is wrong and the slope
becomes an average of two different periods.

Riverbend adopted the de escalation training in July 2023. Fit one line across
the whole period and you get a number that describes neither the period before
nor the period after.

In [ ]:
r = final[final["agency_id"] == "A001"].copy()
r["rate"] = 100 * r["n_uof"] / r["n_arrests"]
r["t"] = np.arange(len(r))
r = r[r["year_month"] <= "2025-12"]

for label, d in [("one line across everything", r),
                 ("before July 2023", r[r["year_month"] < "2023-07"]),
                 ("after July 2023", r[r["year_month"] >= "2023-07"])]:
    e, a, b = slope(d)
    print(f"  {label:28s} {e:+6.2f} percent a year  [{a:+6.2f}, {b:+6.2f}]")

Fitting the right number of lines, and testing where they should break, is
[Module 16](Module_16_Did_Something_Change.md) and Advanced Module 11. For now
the habit is: **before fitting a trend, ask whether anything happened during the
window.**

## 7. What to carry away

| Habit | Why |
|---|---|
| Fit on the log scale | a trend in public safety data is a percentage, not a count |
| Always report the interval | the point estimate alone is misleading precision |
| Use whole calendar years, or add month effects | otherwise the season leaks into the slope |
| Say how long the window is | a two year slope is almost uninformative |
| Check for interventions inside the window | one line assumes one story |

## Exercise

Estimate the trend for Summit County, which the dataset built with a much
steeper decline than everyone else, and check whether your interval covers the
built in value of about 12.2 percent a year.

In [ ]:
# Fill in the blanks, then run.
AGENCY = None              # try "A007"
END = None                 # try "2025-12"

if AGENCY and END:
    d = final[final["agency_id"] == AGENCY].copy()
    d["rate"] = 100 * d["n_uof"] / d["n_arrests"]
    d["t"] = np.arange(len(d))
    d["mon"] = d["year_month"].str[5:7].astype(int)
    d = d[d["year_month"] <= END]
    e, a, b = slope(d, "np.log(rate) ~ t + C(mon)")
    print(f"estimate {e:+.2f} percent a year, interval [{a:+.2f}, {b:+.2f}]")
    print("built in value: -12.19 percent a year")
    print("interval covers it:", a <= -12.19 <= b)
else:
    print("Set AGENCY and END above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A007"
END = "2025-12"
```

The interval covers the built in value. Two things are worth noticing.

First, Summit County's decline is roughly two and a half times the statewide
one, and the estimate separates them clearly. That is what makes A007 the
parallel trends violator in
[Data/GROUND_TRUTH.md](../../../Data/GROUND_TRUTH.md): it was already improving
much faster than everyone else before the training programme began, so including
it in a before and after comparison inflates the apparent effect.

Second, Summit County also adopted the programme, so a single line across the
whole window mixes its own pre existing decline with whatever the programme did.
The number is an accurate description of the window and a poor description of
either half of it.

</details>

---

**Next:** [Module 7, Seasonal Adjustment](Module_07_Seasonal_Adjustment.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*